# Unit Tests for pSeq_Spiral_VDS

This notebook validates trajectory generation and sequence integration for the multishot variable-density spiral readout.

## Covered tests
1. FOV polynomial positivity and shape checks.
2. Gradient and slew compliance with hardware limits.
3. Interleaf rotation behavior.
4. Minimal diffusion sequence timing pass.
5. Timing dictionary contract (`get_timeToTE`).

In [1]:
import numpy as np

import pypulseq as pp
from pSeq import pSeq_Base, pSeq_RF, pSeq_Diffusion, pSeq_Spiral_VDS

In [4]:
def run_tests(verbose=True):
    results = []

    def check(name, fn):
        try:
            fn()
            results.append((name, True, ''))
            if verbose:
                print(f'[PASS] {name}')
        except Exception as exc:
            results.append((name, False, str(exc)))
            if verbose:
                print(f'[FAIL] {name}: {exc}')

    system_max = pp.Opts(
        max_grad=80, grad_unit='mT/m',
        max_slew=150, slew_unit='T/m/s',
        rf_ringdown_time=20e-6, rf_dead_time=100e-6, adc_dead_time=20e-6
    )

    spiral = pSeq_Spiral_VDS(
        system=system_max,
        fov=220e-3,
        nx=128,
        n_interleaves=4,
        fov_coeff=[1.0, 0.0, -0.30],
        oversamp=4,
    )

    # Test 1: FOV profile remains positive for sampled radii
    def test_fov_profile_positive():
        rs = np.linspace(0.0, spiral.kmax, 64)
        vals = np.array([spiral._fov_poly(r)[0] for r in rs])
        if not np.all(vals > 0):
            raise AssertionError('FOV profile has non-positive values.')

    # Test 2: Gradient/slew must satisfy hardware constraints
    def test_grad_slew_limits():
        spiral.prep()
        g_abs = np.abs(spiral.g_traj_hz)
        s_abs = np.abs(spiral.slew_hz)
        if np.max(g_abs) > spiral.system.max_grad * 1.005:
            raise AssertionError('Gradient amplitude exceeds max_grad.')
        if np.max(s_abs) > spiral.system.max_slew * 1.005:
            raise AssertionError(f'Slew rate violation {np.max(s_abs)}')

    # Test 3: Interleaf rotation should match 2*pi/n pattern
    def test_interleaf_rotation():
        if spiral.k_traj is None:
            spiral.prep()

        k0 = spiral.get_ktraj(0)
        k1 = spiral.get_ktraj(1)
        expected = np.exp(1j * 2.0 * np.pi / spiral.n_interleaves)

        err = np.max(np.abs(k1 - expected * k0))
        tol = 5e-4 * spiral.kmax
        if err > tol:
            raise AssertionError(f'Interleaf rotation mismatch: err={err:.3e}, tol={tol:.3e}')

    # Test 4: Spiral readout + rewinder should pass sequence timing
    def test_spiral_block_timing():
        if spiral.k_traj is None:
            spiral.prep()

        seq = pSeq_Base(system=system_max)
        spiral.add_to_seq(seq, interleaf=0, label=False, add_rewinder=True)
        ok, err = seq.seq.check_timing()
        if not ok:
            msg = '; '.join([str(e) for e in err[:3]])
            raise AssertionError(f'Spiral timing failed: {msg}')

    # Test 5: Diffusion designer should integrate with spiral timing model
    def test_diffusion_designer_contract():
        if spiral.k_traj is None:
            spiral.prep()

        rf90 = pSeq_RF(
            system=system_max,
            flip_angle=np.pi / 2,
            duration=3e-3,
            thickness=5e-3,
            use='excitation',
        )
        rf90.prep()

        rf180 = pSeq_RF(
            system=system_max,
            flip_angle=np.pi,
            duration=6e-3,
            thickness=5e-3,
            use='refocusing',
            do_refocus=False,
        )
        rf180.prep()

        diffusion = pSeq_Diffusion(system=system_max, channel='x', target_b_s_mm2=1000, max_lobe_duration=30e-3)
        out = diffusion.prep_monopolar(rf90=rf90, rf180=rf180, epi=spiral, verbose=False)

        if out['b_actual_s_mm2'] <= 0:
            raise AssertionError('Designed diffusion b-value must be positive.')
        if out['delay1'] < 0 or out['delay2'] < 0:
            raise AssertionError('Diffusion delays must be non-negative.')
        if abs(out['spin_echo_mismatch']) > (5 * system_max.grad_raster_time):
            raise AssertionError('Spin-echo mismatch is larger than expected raster tolerance.')

    # Test 6: get_timeToTE contract
    def test_time_to_te_contract():
        t = spiral.get_timeToTE()
        required = {'durationToCenter', 'pre_duration', 'timeToTE', 'ghost_nav'}
        if not required.issubset(set(t.keys())):
            raise AssertionError(f'Missing required keys in get_timeToTE: {required - set(t.keys())}')
        if t['timeToTE'] < 0:
            raise AssertionError('timeToTE must be non-negative.')

    check('FOV profile positivity', test_fov_profile_positive)
    check('Gradient/slew limits', test_grad_slew_limits)
    check('Interleaf rotation', test_interleaf_rotation)
    check('Spiral block timing', test_spiral_block_timing)
    check('Diffusion designer contract', test_diffusion_designer_contract)
    check('get_timeToTE contract', test_time_to_te_contract)

    n_pass = sum(int(ok) for _, ok, _ in results)
    n_all = len(results)
    print(f'\nSummary: {n_pass}/{n_all} tests passed')
    return results

In [5]:
test_results = run_tests(verbose=True)
failed = [r for r in test_results if not r[1]]
if failed:
    raise RuntimeError(f'{len(failed)} tests failed: {failed}')

[PASS] FOV profile positivity
[PASS] Gradient/slew limits
[PASS] Interleaf rotation
[PASS] Spiral block timing
Making Sinc Pulse with flip angle 90.0
Making SLR Pulse with flip angle 90.0
excitation : The peak rf Amplitude =  7.797889527969152 uT
Making Sinc Pulse with flip angle 180.0
Making SLR Pulse with flip angle 180.0
refocusing : The peak rf Amplitude =  7.797889596354154 uT
[PASS] Diffusion designer contract
[PASS] get_timeToTE contract

Summary: 6/6 tests passed
